# Lab 9 — Orchestrator Fan-Out Agent
**Pattern: ORCHESTRATOR-WORKERS** — from HTML Sections 03 and 07

---

## What is the Orchestrator pattern?

One central agent decomposes the task, delegates to specialist workers, and synthesises the results.

The orchestrator does **not** do the work. Workers do. The orchestrator:
1. Reads the request and makes a plan
2. Delegates specific jobs to specialists
3. Receives all outputs
4. Synthesises a final answer

---

## Real-world analogy

A **regional marketing manager** receives a brief: launch a product across five markets simultaneously.

She doesn't translate the copy, write the legal disclaimer, or design the banner.  
She delegates: translator, copywriter, legal team, design team.  
She collects all five outputs and assembles the final package.

That manager is the orchestrator. The specialists are the workers.

---

## Sequential vs Parallel workers

| Type | When to use | Example |
|------|------------|---------|
| **Sequential** | Worker B needs Worker A's output | Researcher → Analyst (analyst needs research data) |
| **Parallel** | Workers are independent | Translate to 5 languages simultaneously |

This lab builds sequential workers. The bonus section at the bottom shows parallel.

---

## What you will learn
1. How to decompose a task dynamically (not hardcoded)
2. How to pass outputs between workers (chaining their results)
3. Why structured JSON output from workers is safer than prose
4. How to run independent workers in parallel using threads
5. Why the orchestrator does a final quality check

## Step 1 — Setup

In [ ]:
import anthropic
import json
import threading
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
print("✓ Client ready")

## Step 2 — Worker Agents

Three workers. Each has one job. Each returns **structured JSON**.

> Why JSON? Because the orchestrator needs to parse and pass outputs between workers reliably.  
> Prose output from a worker forces the next worker to interpret ambiguous text — a source of errors.  
> Structured output is a reliability investment.

In [ ]:
def worker_researcher(topic: str) -> dict:
    """
    Gathers facts. Returns structured JSON.
    Does NOT analyse, does NOT write — only researches.
    """
    print(f"  [RESEARCHER] Starting: {topic[:50]}...")
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system="""You are a specialist research agent. Gather facts only.
Return a JSON object with EXACTLY this structure:
{
  "key_facts":   ["fact 1", "fact 2", "fact 3"],
  "statistics":  ["stat with number 1", "stat with number 2"],
  "key_players": ["entity 1", "entity 2"],
  "timeline":    "brief timeline of relevant events"
}
Return ONLY valid JSON. No preamble.""",
        messages=[{"role": "user", "content": f"Research thoroughly: {topic}"}]
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = "\n".join(raw.split("\n")[1:-1])
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        result = {"key_facts": [raw], "statistics": [], "key_players": [], "timeline": ""}
    print(f"  [RESEARCHER] Done — {len(result.get('key_facts', []))} facts")
    return result


def worker_analyst(topic: str, research_data: dict) -> dict:
    """
    Identifies trends and risks from research data.
    Takes researcher output as input — sequential dependency.
    """
    print(f"  [ANALYST] Analysing: {topic[:50]}...")
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system="""You are a specialist analysis agent. Identify patterns and implications.
Return a JSON object with EXACTLY this structure:
{
  "main_trend":     "one sentence: the dominant trend",
  "opportunities":  ["opportunity 1", "opportunity 2"],
  "risks":          ["risk 1", "risk 2"],
  "recommendation": "one concrete, specific recommendation"
}
Return ONLY valid JSON. No preamble.""",
        messages=[{"role": "user", "content": f"Topic: {topic}\n\nResearch data:\n{json.dumps(research_data, indent=2)}"}]
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = "\n".join(raw.split("\n")[1:-1])
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        result = {"main_trend": raw, "opportunities": [], "risks": [], "recommendation": ""}
    print(f"  [ANALYST] Done")
    return result


def worker_writer(topic: str, research: dict, analysis: dict) -> str:
    """
    Produces the final readable report.
    Receives BOTH research AND analysis — uses outputs from two previous workers.
    """
    print(f"  [WRITER] Writing: {topic[:50]}...")
    context = f"Topic: {topic}\n\nResearch:\n{json.dumps(research, indent=2)}\n\nAnalysis:\n{json.dumps(analysis, indent=2)}"
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1500,
        system="""You are a specialist writing agent. Transform structured data into a clear executive briefing.
Format:
- Opening sentence (the key insight)
- Key Facts (3 bullet points with numbers)
- Analysis (main trend + 2 implications)
- Recommendation (1 specific action)
- Closing (one sentence outlook)

Be specific. Use numbers where available. No jargon.""",
        messages=[{"role": "user", "content": f"Write an executive briefing from:\n\n{context}"}]
    )
    result = response.content[0].text
    print(f"  [WRITER] Done — {len(result.split())} words")
    return result

print("✓ All three workers defined")

## Step 3 — The Orchestrator

The orchestrator coordinates everything. It:
1. **Decomposes** the task dynamically — the subtasks are not hardcoded
2. **Delegates** to each worker in the right sequence
3. **Passes outputs** between workers (research → analyst, both → writer)
4. **Quality checks** the final output before delivery

In [ ]:
class Orchestrator:
    def __init__(self):
        self.execution_log = []

    def _log(self, step: str, detail: str = ""):
        entry = f"[ORCHESTRATOR] {step}" + (f": {detail}" if detail else "")
        self.execution_log.append(entry)
        print(entry)

    def decompose(self, user_request: str) -> dict:
        """
        Dynamic task decomposition — not hardcoded.
        The orchestrator asks Claude to identify the core topic.
        """
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=256,
            system="""You are a task decomposition specialist.
Return a JSON object:
{
  "topic":       "precise research topic in one sentence",
  "focus_areas": ["area 1", "area 2", "area 3"]
}
Return ONLY valid JSON.""",
            messages=[{"role": "user", "content": user_request}]
        )
        raw = response.content[0].text.strip()
        if raw.startswith("```"):
            raw = "\n".join(raw.split("\n")[1:-1])
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            return {"topic": user_request, "focus_areas": []}

    def run(self, user_request: str) -> str:
        print(f"\n{'='*60}")
        print(f"ORCHESTRATOR — Task received")
        print(f"Request: {user_request}")
        print(f"{'='*60}\n")

        # Step 1: Decompose
        self._log("STEP 1", "Decomposing task")
        plan  = self.decompose(user_request)
        topic = plan.get("topic", user_request)
        print(f"  Topic: {topic}")

        # Step 2: Research worker
        self._log("STEP 2", "→ Research Worker")
        research_data = worker_researcher(topic)

        # Step 3: Analysis worker (needs research output)
        self._log("STEP 3", "→ Analysis Worker")
        analysis_data = worker_analyst(topic, research_data)

        # Step 4: Writing worker (needs both outputs)
        self._log("STEP 4", "→ Writing Worker")
        report = worker_writer(topic, research_data, analysis_data)

        # Step 5: Quality check
        self._log("STEP 5", "Quality review")
        review = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=128,
            system="You are a supervisor reviewing a report. Add ONE sentence at the very top starting with 'Quality assessment:'. Then output the full report unchanged.",
            messages=[{"role": "user", "content": report}]
        )
        final = review.content[0].text
        self._log("COMPLETE", f"{len(final.split())} word report delivered")
        return final

    def show_log(self):
        print("\n── Orchestration Log ──")
        for entry in self.execution_log:
            print(f"  {entry}")

print("✓ Orchestrator defined")

## Step 4 — Run the Orchestrator

In [ ]:
orchestrator = Orchestrator()
final_report = orchestrator.run(
    "What is the current state of edge computing and why should enterprise architects care?"
)

In [ ]:
print("=" * 60)
print("FINAL DELIVERED REPORT:")
print("=" * 60)
print(final_report)
orchestrator.show_log()

## Bonus: Parallel Workers

When workers are **independent** (don't need each other's output), run them simultaneously.

Total time ≈ one API call. Not three sequential calls.  
This is the **parallelisation** pattern from HTML Section 07.

In [ ]:
def run_parallel_workers(topics: list) -> dict:
    """
    Three independent research workers running simultaneously.
    Each gets its own thread. Results collected when all finish.
    
    Real-world parallel: running unit tests, integration tests,
    and security scans at the same time in your CI pipeline.
    """
    results = {}
    threads = []

    def research_wrapper(topic):
        data = worker_researcher(topic)
        results[topic] = data.get("key_facts", ["No data"])[0] if data.get("key_facts") else "No data"

    print(f"\n── Starting {len(topics)} parallel workers simultaneously ──")
    for topic in topics:
        t = threading.Thread(target=research_wrapper, args=(topic,))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    return results

parallel_results = run_parallel_workers([
    "Kubernetes security hardening for financial services",
    "AWS cost optimisation strategies for EKS clusters",
    "AI inference hardware trends: GPUs vs custom silicon",
])

print("\n── Parallel results collected ──")
for topic, result in parallel_results.items():
    print(f"  [{topic[:45]}]")
    print(f"   → {result[:90]}...")

## Key Takeaways

| Point | Why it matters |
|-------|---------------|
| Orchestrator plans, workers execute | Workers can be swapped without touching the orchestrator |
| Structured JSON between workers | Reliable parsing. No ambiguity. Gates can validate it programmatically. |
| Sequential when outputs chain | Researcher → Analyst is sequential because analyst needs research data |
| Parallel when outputs are independent | Translate to 5 languages: all 5 run at the same time |
| Quality check as a final gate | Orchestrator can catch and fix poor worker output before delivery |

---

**Next lab:** Lab 10 — Prompt Chaining with Verification Gates  
A four-step pipeline where programmatic gates between steps catch errors before they propagate.